# Iris with micrograd

Take the autograd engine from Day 1/2 and train a real classifier on the **Iris** dataset:

- 4 features (sepal length, sepal width, petal length, petal width)
- 3 classes (setosa, versicolor, virginica)
- 150 samples, 80/20 stratified split

Target: **>=90% test accuracy** in ~200 steps. If you hit that, the autograd engine works on a real-world tabular problem, not just XOR.

## Things you'll notice (and one bug you'll fix)

1. **Multi-class output** — 3 logits + softmax + cross-entropy. We add a `log` op to micrograd inline because the solo file didn't include one.
2. **The `MLP` in `micrograd_solo.py` applies `tanh` to every layer including the output.** For a classifier, that saturates the logits to [-1, 1] and crushes the softmax's ability to make confident predictions. We define our own MLP here where the **last layer is linear**.
3. **Mini-batches?** No. Pure-scalar micrograd makes that prohibitive. Full-batch SGD instead. Wall time: ~30-60 sec for 200 steps. This is exactly why `02_makemore/` moves to tensor ops next.

In [ ]:
import math
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Find micrograd_solo.py next to this notebook regardless of where VS Code launched.
NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR / 'experiments' / '01_micrograd']:
    if (candidate / 'micrograd_solo.py').exists():
        sys.path.insert(0, str(candidate))
        break
from micrograd_solo import Value

random.seed(42)
np.random.seed(42)
print('imports ok')

## Data

Iris is the boring classic. The point isn't novelty — it's that **micrograd handles it without modification to the autograd engine**.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'classes: {dict(zip(iris.target_names, range(3)))}')
print(f'feature ranges (raw):  min {X.min(axis=0)}   max {X.max(axis=0)}')

X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42,
)
print(f'\ntrain: {len(X_train)} examples,  test: {len(X_test)} examples')

## Loss: softmax + cross-entropy

`Value` doesn't have `.log()` (we didn't need it for XOR). We add a one-off below — `d/dv log(v) = 1/v`.

Numerical detail: naive `exp(logit)` overflows for large logits. The standard trick is `softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))`. Subtracting the max is mathematically a no-op but keeps `exp()` inputs bounded.

In [ ]:
def log_value(v):
    """log(v) as a one-off Value op. d/dv = 1/v."""
    out = Value(math.log(v.data), (v,), 'log')
    def _backward():
        v.grad += (1.0 / v.data) * out.grad
    out._backward = _backward
    return out


def softmax(logits):
    m = max(l.data for l in logits)            # plain float, used as constant
    shifted = [(l - m) for l in logits]
    exps = [s.exp() for s in shifted]
    total = exps[0]
    for e in exps[1:]:
        total = total + e
    return [e / total for e in exps]


def cross_entropy(logits, target_idx):
    probs = softmax(logits)
    return -log_value(probs[target_idx])


# Sanity: at uniform probabilities the CE loss should be log(3).
random.seed(0)
test_logits = [Value(random.uniform(-2, 2)) for _ in range(3)]
test_loss = cross_entropy(test_logits, target_idx=1)
print(f'random-init CE loss, target=1: {test_loss.data:.4f}')
print(f'(at uniform probs the loss is log(3) = {math.log(3):.4f})')

## Model: 4 → 8 → 3 with a linear output layer

**Why not use `MLP` from `micrograd_solo.py`?**

Because that class applies `tanh` to every layer including the last. For a classifier, the output should be **raw logits** so softmax can express any distribution. If you tanh the logits first, they're squished into [-1, 1] and softmax can't differentiate confident predictions from middling ones — you class-collapse and stall at ~67% accuracy.

Real lesson: **activation functions are about the loss family, not just "deeper is better"**. With cross-entropy + softmax, the last layer is linear by convention. With MSE for regression on bounded targets, you'd tanh the output.

In [ ]:
class LinearLayer:
    """out_j = sum_i(w_ji * x_i) + b_j   (no nonlinearity)."""
    def __init__(self, nin, nout):
        self.w = [[Value(random.uniform(-1, 1)) for _ in range(nin)] for _ in range(nout)]
        self.b = [Value(0.0) for _ in range(nout)]

    def __call__(self, x):
        return [sum((wij * xj for wij, xj in zip(wi, x)), bi)
                for wi, bi in zip(self.w, self.b)]

    def parameters(self):
        return [p for wi in self.w for p in wi] + self.b


class TanhLayer(LinearLayer):
    """Linear + tanh on each output. For hidden layers."""
    def __call__(self, x):
        return [v.tanh() for v in super().__call__(x)]


class MLP:
    """Hidden TanhLayers ending in a single LinearLayer (the logits layer)."""
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = []
        for i in range(len(nouts) - 1):
            self.layers.append(TanhLayer(sizes[i], sizes[i + 1]))
        self.layers.append(LinearLayer(sizes[-2], sizes[-1]))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for L in self.layers for p in L.parameters()]


random.seed(42)
model = MLP(nin=4, nouts=[8, 3])
n_params = len(model.parameters())
print(f'parameters: {n_params}  ({4*8 + 8} hidden + {8*3 + 3} output)')

## Train

Full-batch SGD: zero_grad → forward → loss → backward → step.

Wall time: ~30-60 seconds for 200 steps. The slow part is pure-Python autograd (~120 examples × ~50 scalar ops × 200 steps ≈ 1.2M Python ops).

In [ ]:
def predict_class(model, x_row):
    logits = model([Value(v) for v in x_row])
    return max(range(3), key=lambda i: logits[i].data)

def accuracy(model, X, y):
    return sum(predict_class(model, X[i]) == y[i] for i in range(len(X))) / len(X)

lr = 0.05
steps = 200
history = []  # (step, mean_loss, train_acc, test_acc)
t0 = time.time()

for step in range(steps + 1):
    # ---- forward + losses ----
    losses = []
    for i in range(len(X_train)):
        logits = model([Value(v) for v in X_train[i]])
        losses.append(cross_entropy(logits, y_train[i]))
    total_loss = losses[0]
    for L in losses[1:]:
        total_loss = total_loss + L
    mean_loss = total_loss / len(losses)

    # ---- zero grad, backward, step ----
    for p in model.parameters():
        p.grad = 0.0
    mean_loss.backward()
    for p in model.parameters():
        p.data -= lr * p.grad

    if step % 25 == 0:
        tr = accuracy(model, X_train, y_train)
        te = accuracy(model, X_test,  y_test)
        history.append((step, mean_loss.data, tr, te))
        print(f'step {step:3d}  loss {mean_loss.data:.4f}  train {tr:.3f}  test {te:.3f}  ({time.time()-t0:.1f}s)')

## Results plot

In [ ]:
steps_x = [h[0] for h in history]
losses_y = [h[1] for h in history]
tr_y = [h[2] for h in history]
te_y = [h[3] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps_x, losses_y, marker='o', color='#cf222e', linewidth=1.6)
ax1.set_xlabel('step'); ax1.set_ylabel('mean cross-entropy')
ax1.set_title('Train loss'); ax1.grid(True, alpha=0.3)

ax2.plot(steps_x, tr_y, marker='o', label='train', color='#0969da', linewidth=1.6)
ax2.plot(steps_x, te_y, marker='s', label='test',  color='#cf222e', linewidth=1.6)
ax2.set_xlabel('step'); ax2.set_ylabel('accuracy')
ax2.set_title('Accuracy'); ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3); ax2.legend()

fig.tight_layout()
out = Path('runs') / 'iris_loss.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=130)
print(f'wrote {out}')
plt.show()

print(f'\nfinal train acc: {tr_y[-1]:.3f}')
print(f'final test  acc: {te_y[-1]:.3f}')
if te_y[-1] >= 0.90:
    print('Target hit. micrograd trains a real classifier.')
else:
    print(f'Below 0.90. Try: more steps, higher LR, or check the architecture.')

## Honest reflection

**What this proves:** the autograd engine handles a real ML task without modification. Same `Value`, same `backward()`, same SGD pattern as XOR — only the loss and the output-layer activation changed.

**The bug we found:** `MLP` in `micrograd_solo.py` hardcodes tanh on every layer including the output. For classification with softmax + cross-entropy that's wrong — it caps test accuracy at ~67%. Fix is one architectural change (linear output layer); the autograd engine itself is fine.

**The cost:** ~50 seconds for 200 steps on 67 parameters and 120 training examples. Compare to PyTorch on the same problem (~0.5 sec). The 100x slowdown is what makes `02_makemore/` necessary — it teaches you to run autograd on **tensors** (groups of numbers at once) instead of scalars.

**One line for `NOTES.md`:** what was the most surprising thing about training on real data with the scalar engine?